### code
dd

### Import et install

In [2]:
import pandas as pd
import json
import os
import sys

ModuleNotFoundError: No module named 'pandas'

chargement data


In [ ]:

# Chargement du fichier
fichier = "DATA\sample_queries.json"
with open(fichier, "r", encoding="utf-8") as f:
    data = json.load(f)

# ── Option 1 : DataFrame simple (1 ligne par question) ──────────────────────
df_annot = pd.DataFrame(data["results"])

In [ ]:
df_annot.head()

,qid,question,retrieved,answer,metadata
0,Q1,"Quel est l'objectif du projet ""Beehive"" de la ...","[{'rank': 1, 'doc_name': '20260108_NP_Obsdrone...","Le projet **""Beehive""** de la Royal Navy a pou...",{'key': 'value'}
1,Q2,Comment l’intégration du drone MQ-9 Reaper dan...,"[{'rank': 1, 'doc_name': 'r20-7111.pdf', 'page...",L’intégration du **MQ-9 Reaper** dans l’Armée ...,{'key': 'value'}
2,Q3,Comment les mesures anti-drones proposées en F...,"[{'rank': 1, 'doc_name': 'l15b4320_rapport-inf...",Il existe une **tendance mondiale vers l’intég...,{'key': 'value'}
3,Q4,Comment les méthodes de renseignement telles q...,"[{'rank': 1, 'doc_name': 'open-source-intellig...",Pour renforcer la détection des circuits de fi...,{'key': 'value'}
4,Q5,Quels sont les drones capables de transporter ...,"[{'rank': 1, 'doc_name': 'TODO', 'page': 42, '...",TODO,{'key': 'value'}


### Mise en format de lecture 


In [ ]:
import os
import re
import platform
import pandas as pd
import pytesseract
from pathlib import Path
from tempfile import TemporaryDirectory
from pdf2image import convert_from_path
from PIL import Image

if platform.system() == "Windows":
    pytesseract.pytesseract.tesseract_cmd = r"C:/Program Files/Tesseract-OCR/tesseract.exe"
    path_to_poppler_exe = Path(r"C:/Users/natal/OneDrive\Bureau/COURS_TELECOM/Amiad Hackathon/utilitaire/poppler-26.02.0/library/bin") 


def pdf_to_text_ocr(pdf_path: Path) -> str:
    with TemporaryDirectory() as tempdir:
        if platform.system() == "Windows":
            pdf_pages = convert_from_path(pdf_path, 300, poppler_path=path_to_poppler_exe)
        else:
            pdf_pages = convert_from_path(pdf_path, 300)

        full_text = ""
        for i, page in enumerate(pdf_pages, start=1):
            img_path = f"{tempdir}/page_{i:03}.jpg"
            page.save(img_path, "JPEG")
            text = pytesseract.image_to_string(Image.open(img_path), lang="fra")
            text = text.replace("-\n", "")
            full_text += text + "\n"

    return full_text


def split_into_paragraphs(text: str) -> list[str]:
    paragraphs = re.split(r"\n\s*\n", text)
    cleaned = []
    for para in paragraphs:
        para = re.sub(r"(?<!\n)\n(?!\n)", " ", para).strip()
        para = re.sub(r" {2,}", " ", para)
        if len(para) > 50:
            cleaned.append(para)
    return cleaned


def build_paragraphs_dataframe(pdf_folder: str) -> pd.DataFrame:
    pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]

    if not pdf_files:
        print(f"Aucun fichier PDF trouvé dans : {pdf_folder}")
        return pd.DataFrame(columns=["paragraph", "source_file"])

    rows = []
    for filename in pdf_files:
        pdf_path = Path(pdf_folder) / filename
        print(f"Traitement OCR : {filename}")

        try:
            text = pdf_to_text_ocr(pdf_path)
            paragraphs = split_into_paragraphs(text)
            for para in paragraphs:
                rows.append({"paragraph": para, "source_file": filename})
            print(f"  → {len(paragraphs)} paragraphes extraits")

        except Exception as e:
            print(f"  ⚠ Erreur sur {filename} : {e}")

    df = pd.DataFrame(rows, columns=["paragraph", "source_file"])
    print(f"\nTotal : {len(df)} paragraphes depuis {len(pdf_files)} fichiers.")
    return df


# ── Utilisation ──────────────────────────────────────────────────────────────

PDF_FOLDER = "DATA\\Corpus_raw"  

df_paragraphs = build_paragraphs_dataframe(PDF_FOLDER)

print(df_paragraphs.head(10))



Traitement OCR : 04500463_rev_e_smdr_feature_ovvu.pdf
  → 79 paragraphes extraits
Traitement OCR : 10e réunion du GT drones _ Ministère des Armées et des Anciens combattants.pdf
  → 23 paragraphes extraits
Traitement OCR : 116_4_filiere-hp_dronisation_web_0.pdf
  → 217 paragraphes extraits
Traitement OCR : 14 juillet 2025 _ trois raisons de s'intéresser au robot mule PROBOT.pdf
  → 29 paragraphes extraits
Traitement OCR : 1503_651674.pdf
  → 183 paragraphes extraits
Traitement OCR : 171110_np_arm-dpid_286-no-homologation-des-systemes-de-protection-de-site.pdf
  → 35 paragraphes extraits
Traitement OCR : 1740_install.pdf
  → 24 paragraphes extraits
Traitement OCR : 2005_rapport_senat_r05-2151.pdf
  → 436 paragraphes extraits
Traitement OCR : 20081006-np-cicde-pia-3312-drones-2008.pdf
  → 232 paragraphes extraits
Traitement OCR : 2011113_IGI 1300_Protection du secret de la defense nationale_NP.pdf
  → 1771 paragraphes extraits
Traitement OCR : 20120606-NP-CICDE-RDIA-2012-010-ESDA-2012_fu

c:\Users\natal\OneDrive\Bureau\COURS_TELECOM\Amiad Hackathon\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (133930126 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  → 4 paragraphes extraits
Traitement OCR : 20231019_NP_Arrêté-liste-2023-ZICAD.pdf
  → 1101 paragraphes extraits
Traitement OCR : 20231220_Vocabulaire militaire français-anglais.pdf
  → 2280 paragraphes extraits
Traitement OCR : 20231222_NP_DMCA Bilan de l'année 2023.pdf
  → 42 paragraphes extraits
Traitement OCR : 20240126_NP_DRHMD-PRH1_GUIDE-metiers-REM-2025.pdf
  → 1003 paragraphes extraits
Traitement OCR : 20240131_np_dpid_dir33-directive-retour-experience-en-securite-numerique.pdf
  → 53 paragraphes extraits
Traitement OCR : 20240416_NP_NOTE_D-24-001879_directive de pilotage budgétaire 2024.pdf
  → 997 paragraphes extraits
Traitement OCR : 20240419_NP_Livret_accueil_DGA_MI.pdf
  → 189 paragraphes extraits
Traitement OCR : 20240429_NP_DRH-MD-PRH1_PLAQUETTE-COM-REM-2025-PUBLICATION-SGA-CONNECT.pdf
  → 464 paragraphes extraits
Traitement OCR : 20241202_NP_SGDSN_VIGINUM_RAPPORT-BIG.pdf
  → 234 paragraphes extraits
Traitement OCR : 2025-Analysis-of-Russian-Shahed-type-UAVs-Deployment-

In [ ]:
# Charger le fichier Excel dans un DataFrame
df_paragraphs.to_csv('fichier.csv', index=False)


In [ ]:
df_paragraphs.head()

,paragraph,source_file
0,"On ESI systems, station message detail reporti...",04500463_rev_e_smdr_feature_ovvu.pdf
1,Virtually all call accounting systems marketed...,04500463_rev_e_smdr_feature_ovvu.pdf
2,"SMDR may be output in one of three formats, se...",04500463_rev_e_smdr_feature_ovvu.pdf
3,You can store SMDR records if using either (a....,04500463_rev_e_smdr_feature_ovvu.pdf
4,Stored SMDR records can be exported to the .CS...,04500463_rev_e_smdr_feature_ovvu.pdf


### Fine tuning pour vectorisation 

In [ ]:
import json
import pandas as pd
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
from sentence_transformers import SentenceTransformer


# Données 

def load_training_pairs(df_paragraphs: pd.DataFrame, json_path: str) -> pd.DataFrame:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for result in data["results"]:
        question = result["question"]
        for doc in result["retrieved"]:
            matching = df_paragraphs[
                df_paragraphs["source_file"] == doc["doc_name"]
            ]["paragraph"].tolist()
            for para in matching:
                rows.append({"question": question, "paragraph": para, "source_file": doc["doc_name"]})

    return pd.DataFrame(rows)


#  Dataset ───────────────────────────────────────────────────────────────

class MultiPositiveDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.questions  = df["question"].tolist()
        self.paragraphs = df["paragraph"].tolist()
        self.sources    = df["source_file"].tolist()

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return {
            "question":  "query: "   + self.questions[idx],
            "paragraph": "passage: " + self.paragraphs[idx],
            "source":    self.sources[idx],
        }


def collate_fn(batch):
    return {
        "questions":  [b["question"]  for b in batch],
        "paragraphs": [b["paragraph"] for b in batch],
        "sources":    [b["source"]    for b in batch],
    }


#  Encode avec gradient 

def encode_with_grad(
    model: SentenceTransformer,
    texts: list[str],
    device: torch.device,
) -> torch.Tensor:
    """
    Forward natif avec gradients actifs.
    FIX 1 : évite model.encode() qui wrappe dans torch.no_grad()
    FIX 2 : filtre les valeurs non-Tensor avant .to(device)
    """
    features = model.tokenize(texts)
    features = {
        k: v.to(device) if isinstance(v, torch.Tensor) else v
        for k, v in features.items()
    }
    out = model(features)
    emb = out["sentence_embedding"]
    return F.normalize(emb, p=2, dim=-1)


#  Loss multi-positifs 

def multi_positive_loss(
    q_emb: torch.Tensor,
    p_emb: torch.Tensor,
    sources: list[str],
    temperature: float = 0.07,
) -> torch.Tensor:
    logits = torch.matmul(q_emb, p_emb.T) / temperature

    positive_mask = torch.tensor(
        [[sources[i] == sources[j] for j in range(len(sources))] for i in range(len(sources))],
        dtype=torch.float,
        device=q_emb.device,
    )
    labels = positive_mask / positive_mask.sum(dim=1, keepdim=True)

    loss_q = -(labels   * F.log_softmax(logits,   dim=1)).sum(dim=1).mean()
    loss_p = -(labels.T * F.log_softmax(logits.T, dim=1)).sum(dim=1).mean()
    return (loss_q + loss_p) / 2


# Fine-tuning 

def fine_tune(
    df_pairs:    pd.DataFrame,
    model_name:  str   = "intfloat/multilingual-e5-large",
    output_dir:  str   = "models/e5-finetuned",
    epochs:      int   = 3,
    batch_size:  int   = 32,
    lr:          float = 2e-5,
    temperature: float = 0.07,
) -> SentenceTransformer:

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device : {device}")

    if device.type == "cuda":
        print(f"GPU    : {torch.cuda.get_device_name(0)}")
        print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    #  Modèle 
    model = SentenceTransformer(model_name).to(device)

    # FIX OOM : gradient checkpointing — réduit la VRAM ~30% au prix d'un
    # léger surcoût CPU (recalcul des activations intermédiaires)
    model[0].auto_model.gradient_checkpointing_enable()
    print("Gradient checkpointing : activé")

    # ── DataLoader ────────────────────────────────────────────────────────────
    dataset    = MultiPositiveDataset(df_pairs)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        pin_memory=(device.type == "cuda"),  
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    #  mixed precision fp16
    use_amp = device.type == "cuda"
    scaler  = GradScaler(enabled=use_amp)
    print(f"Mixed precision fp16 : {'activé' if use_amp else 'désactivé (CPU)'}")

    # entrainement
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for step, batch in enumerate(dataloader):

            with autocast(enabled=use_amp):
                q_emb = encode_with_grad(model, batch["questions"],  device)
                p_emb = encode_with_grad(model, batch["paragraphs"], device)
                loss  = multi_positive_loss(q_emb, p_emb, batch["sources"], temperature)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            if step % 10 == 0:
                vram = torch.cuda.memory_allocated() / 1e9 if use_amp else 0
                print(f"  Epoch {epoch+1} | Step {step:4d} | Loss {loss.item():.4f} | VRAM {vram:.1f} GB")

        avg = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs} — Loss moyenne : {avg:.4f}")

    model.save(output_dir)
    print(f"Modèle sauvegardé : {output_dir}")
    return model


# main

if __name__ == "__main__":
    df_pairs = load_training_pairs(df_paragraphs, json_path="DATA/sample_queries.json")
    model    = fine_tune(df_pairs, batch_size=32)

Device : cpu


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2373.42it/s]


  Epoch 1 | Step    0 | Loss 3.4263


: 